# Análise das variáveis — Ativação de Sellers (Olist)

Notebook de **demonstração**: roda cada variável de `variaveis.md` contra o banco de validação local **`olist.db`** (SQLite) e mostra a tabela resultante (`seller_id` × colunas da feature).

Cada seção tem: a **explicação** (markdown) e o **SQL** (célula de código) — o mesmo SQL validado em `features_sqlite/`. Os scripts de produção em Spark estão em `features_spark/` (`workspace.olist.*`).

**Premissas-chave:** data de venda = `order_purchase_timestamp` (corte estrito `< {data_corte}`); **sem** filtro de `order_status`; grão = 1 linha por `seller_id`. Detalhes em [`docs/variaveis_detalhadas.md`](docs/variaveis_detalhadas.md).

👉 Para trocar a data de corte, edite `DATA_CORTE` na célula de setup e rode tudo.

In [ ]:
# === Setup: conexão, parâmetro de corte e helper ===
import os, sys, subprocess, sqlite3
import pandas as pd

DB = 'olist.db'
DATA_CORTE = '2018-09-01'   # <<< parâmetro de corte (altere aqui e rode tudo)

# gera o banco a partir de dados/ caso ainda não exista
if not os.path.exists(DB):
    subprocess.run([sys.executable, 'scripts/build_sqlite.py'], check=True)

con = sqlite3.connect(DB)

def run_sql(sql: str) -> pd.DataFrame:
    """Substitui {data_corte} pelo parâmetro e retorna o resultado como DataFrame."""
    return pd.read_sql_query(sql.replace('{data_corte}', DATA_CORTE), con)

print('Banco:', DB, '| data_corte =', DATA_CORTE)

## 1. Categorias distintas no período  `*`

Quantas **categorias diferentes** o seller vendeu em cada janela (`D28/D56/D365/Vida`). Categoria sem cadastro vira `'sem_categoria'`. Mede a **diversidade** do portfólio.

In [ ]:
# Feature 01_qtd_categorias_distintas  (fonte: features_sqlite/01_qtd_categorias_distintas.sql)
SQL = """
-- =====================================================================
-- Feature 01 (variaveis.md item 1) — Quantidade de categorias distintas
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Definição : nº de categorias distintas (product_category_name) de
--             produtos VENDIDOS pelo seller em cada janela.
-- Data venda: order_purchase_timestamp (corte estrito < {data_corte}).
-- Premissas : sem filtro de order_status (todo pedido conta);
--             categoria NULL -> 'sem_categoria' (não some no DISTINCT).
-- Parâmetro : {data_corte} (substituir pela data de corte; ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
        o.order_purchase_timestamp                         AS dt_venda
    FROM order_items oi
    JOIN orders      o ON o.order_id   = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'      -- Vida (corte estrito)
)
SELECT
    seller_id,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-28 days')
                        THEN categoria END) AS qtd_categorias_distintas_d28,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-56 days')
                        THEN categoria END) AS qtd_categorias_distintas_d56,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-365 days')
                        THEN categoria END) AS qtd_categorias_distintas_d365,
    COUNT(DISTINCT categoria)               AS qtd_categorias_distintas_vida
FROM vendas
GROUP BY seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 2. Produtos distintos no período  `*`

Quantos **`product_id` diferentes** o seller vendeu por janela. Amplitude do catálogo efetivamente vendido.

In [ ]:
# Feature 02_qtd_produtos_distintos  (fonte: features_sqlite/02_qtd_produtos_distintos.sql)
SQL = """
-- =====================================================================
-- Feature 02 (variaveis.md item 2) — Quantidade de produtos distintos
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Definição : nº de product_id distintos VENDIDOS pelo seller na janela.
-- Data venda: order_purchase_timestamp (corte estrito < {data_corte}).
-- Premissas : sem filtro de order_status; "produto" = SKU (product_id).
-- Parâmetro : {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.product_id,
        o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT
    seller_id,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-28 days')
                        THEN product_id END) AS qtd_produtos_distintos_d28,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-56 days')
                        THEN product_id END) AS qtd_produtos_distintos_d56,
    COUNT(DISTINCT CASE WHEN dt_venda >= datetime('{data_corte}', '-365 days')
                        THEN product_id END) AS qtd_produtos_distintos_d365,
    COUNT(DISTINCT product_id)               AS qtd_produtos_distintos_vida
FROM vendas
GROUP BY seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 3. Concorrentes na mesma categoria  `*`

Para cada seller, quantos **outros** sellers venderam em **alguma categoria em comum**, na mesma janela. Concorrência **indireta**.

In [ ]:
# Feature 03_concorrentes_mesma_categoria  (fonte: features_sqlite/03_concorrentes_mesma_categoria.sql)
SQL = """
-- =====================================================================
-- Feature 03 (variaveis.md item 3) — Sellers concorrentes em mesma categoria
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Definição : para o seller A, nº de OUTROS sellers (B<>A) que venderam
--             em ALGUMA categoria em que A vendeu, na mesma janela.
--             (concorrência indireta — produtos substitutos).
-- Data venda: order_purchase_timestamp (corte estrito < {data_corte}).
-- Premissas : sem filtro de order_status; categoria NULL -> 'sem_categoria';
--             medida simétrica; sem venda na janela -> 0 concorrentes.
-- Auto-crítica: o DISTINCT (seller,categoria) por janela + COUNT(DISTINCT
--             b.seller_id) impede dupla contagem mesmo quando A e B
--             compartilham várias categorias.
-- Parâmetro : {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
        o.order_purchase_timestamp                         AS dt_venda
    FROM order_items oi
    JOIN orders      o ON o.order_id   = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
-- pares distintos (seller, categoria) ativos em cada janela
sc_d28  AS (SELECT DISTINCT seller_id, categoria FROM vendas WHERE dt_venda >= datetime('{data_corte}', '-28 days')),
sc_d56  AS (SELECT DISTINCT seller_id, categoria FROM vendas WHERE dt_venda >= datetime('{data_corte}', '-56 days')),
sc_d365 AS (SELECT DISTINCT seller_id, categoria FROM vendas WHERE dt_venda >= datetime('{data_corte}', '-365 days')),
sc_vida AS (SELECT DISTINCT seller_id, categoria FROM vendas),
-- concorrentes distintos por janela
conc_d28  AS (SELECT a.seller_id, COUNT(DISTINCT b.seller_id) AS q FROM sc_d28  a JOIN sc_d28  b ON b.categoria = a.categoria AND b.seller_id <> a.seller_id GROUP BY a.seller_id),
conc_d56  AS (SELECT a.seller_id, COUNT(DISTINCT b.seller_id) AS q FROM sc_d56  a JOIN sc_d56  b ON b.categoria = a.categoria AND b.seller_id <> a.seller_id GROUP BY a.seller_id),
conc_d365 AS (SELECT a.seller_id, COUNT(DISTINCT b.seller_id) AS q FROM sc_d365 a JOIN sc_d365 b ON b.categoria = a.categoria AND b.seller_id <> a.seller_id GROUP BY a.seller_id),
conc_vida AS (SELECT a.seller_id, COUNT(DISTINCT b.seller_id) AS q FROM sc_vida a JOIN sc_vida b ON b.categoria = a.categoria AND b.seller_id <> a.seller_id GROUP BY a.seller_id),
spine AS (SELECT DISTINCT seller_id FROM vendas)
SELECT
    s.seller_id,
    COALESCE(conc_d28.q,  0) AS qtd_concorrentes_mesma_categoria_d28,
    COALESCE(conc_d56.q,  0) AS qtd_concorrentes_mesma_categoria_d56,
    COALESCE(conc_d365.q, 0) AS qtd_concorrentes_mesma_categoria_d365,
    COALESCE(conc_vida.q, 0) AS qtd_concorrentes_mesma_categoria_vida
FROM spine s
LEFT JOIN conc_d28  ON conc_d28.seller_id  = s.seller_id
LEFT JOIN conc_d56  ON conc_d56.seller_id  = s.seller_id
LEFT JOIN conc_d365 ON conc_d365.seller_id = s.seller_id
LEFT JOIN conc_vida ON conc_vida.seller_id = s.seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 4. Concorrentes no mesmo produto  `*`

Quantos **outros** sellers venderam o **mesmo `product_id`**, por janela. Concorrência **direta** (mesmo SKU → guerra de preço).

In [ ]:
# Feature 04_concorrentes_mesmo_produto  (fonte: features_sqlite/04_concorrentes_mesmo_produto.sql)
SQL = """
-- =====================================================================
-- Feature 04 (variaveis.md item 4) — Sellers concorrentes no mesmo produto
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Definição : para o seller A, nº de OUTROS sellers (B<>A) que venderam
--             ao menos um dos MESMOS product_id que A, na mesma janela.
--             (concorrência DIRETA — mesmo SKU = guerra de preço).
-- Data venda: order_purchase_timestamp (corte estrito < {data_corte}).
-- Premissas : sem filtro de order_status; product_id sempre presente.
-- Auto-crítica: DISTINCT (seller, product_id) + COUNT(DISTINCT b.seller_id)
--             evita dupla contagem.
-- Parâmetro : {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.product_id,
        o.order_purchase_timestamp AS dt_venda
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
sp_d28  AS (SELECT DISTINCT seller_id, product_id FROM vendas WHERE dt_venda >= datetime('{data_corte}', '-28 days')),
sp_d56  AS (SELECT DISTINCT seller_id, product_id FROM vendas WHERE dt_venda >= datetime('{data_corte}', '-56 days')),
sp_d365 AS (SELECT DISTINCT seller_id, product_id FROM vendas WHERE dt_venda >= datetime('{data_corte}', '-365 days')),
sp_vida AS (SELECT DISTINCT seller_id, product_id FROM vendas),
conc_d28  AS (SELECT a.seller_id, COUNT(DISTINCT b.seller_id) AS q FROM sp_d28  a JOIN sp_d28  b ON b.product_id = a.product_id AND b.seller_id <> a.seller_id GROUP BY a.seller_id),
conc_d56  AS (SELECT a.seller_id, COUNT(DISTINCT b.seller_id) AS q FROM sp_d56  a JOIN sp_d56  b ON b.product_id = a.product_id AND b.seller_id <> a.seller_id GROUP BY a.seller_id),
conc_d365 AS (SELECT a.seller_id, COUNT(DISTINCT b.seller_id) AS q FROM sp_d365 a JOIN sp_d365 b ON b.product_id = a.product_id AND b.seller_id <> a.seller_id GROUP BY a.seller_id),
conc_vida AS (SELECT a.seller_id, COUNT(DISTINCT b.seller_id) AS q FROM sp_vida a JOIN sp_vida b ON b.product_id = a.product_id AND b.seller_id <> a.seller_id GROUP BY a.seller_id),
spine AS (SELECT DISTINCT seller_id FROM vendas)
SELECT
    s.seller_id,
    COALESCE(conc_d28.q,  0) AS qtd_concorrentes_mesmo_produto_d28,
    COALESCE(conc_d56.q,  0) AS qtd_concorrentes_mesmo_produto_d56,
    COALESCE(conc_d365.q, 0) AS qtd_concorrentes_mesmo_produto_d365,
    COALESCE(conc_vida.q, 0) AS qtd_concorrentes_mesmo_produto_vida
FROM spine s
LEFT JOIN conc_d28  ON conc_d28.seller_id  = s.seller_id
LEFT JOIN conc_d56  ON conc_d56.seller_id  = s.seller_id
LEFT JOIN conc_d365 ON conc_d365.seller_id = s.seller_id
LEFT JOIN conc_vida ON conc_vida.seller_id = s.seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 5. Estatísticas de caracteres da descrição  (Vida)

Média, p25, mediana, p75, min e max do tamanho da descrição dos **produtos distintos** vendidos. Proxy de **qualidade do cadastro**.

In [ ]:
# Feature 05_desc_chars_estatisticas  (fonte: features_sqlite/05_desc_chars_estatisticas.sql)
SQL = """
-- =====================================================================
-- Feature 05 (variaveis.md item 5) — Estatísticas de caracteres da descrição
-- Janela: Vida (sem '*')           |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : desc_chars_media, desc_chars_p25, desc_chars_mediana (p50),
--            desc_chars_p75, desc_chars_min, desc_chars_max.
-- Definição: estatísticas de product_description_lenght sobre os PRODUTOS
--            DISTINTOS vendidos pelo seller (cada produto pesa 1 — é
--            propriedade de cadastro, ver premissa 3.6).
-- Data     : order_purchase_timestamp < {data_corte}.
-- ---------------------------------------------------------------------
-- Percentil: SQLite não tem percentile(); calculamos o percentil
--            CONTÍNUO tipo-7 (interpolação linear, idêntico ao
--            percentile() do Spark): idx = (n-1)*p; valor = v[floor(idx)]
--            + frac*(v[floor(idx)+1] - v[floor(idx)]).
-- Nulos    : descrição NULL é ignorada; seller sem produto com descrição
--            -> todas as colunas NULL.
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (   -- todas as vendas Vida do seller (define o universo/spine)
    SELECT
        oi.seller_id,
        oi.product_id,
        p.product_description_lenght AS L
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
prod AS (   -- produtos DISTINTOS com descrição (cada produto pesa 1)
    SELECT DISTINCT seller_id, product_id, L
    FROM vendas
    WHERE L IS NOT NULL
),
ranked AS (
    SELECT
        seller_id, L,
        CAST(ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY L) AS INTEGER) - 1 AS rn0,
        COUNT(*) OVER (PARTITION BY seller_id) AS n
    FROM prod
),
agg AS (
    SELECT
        seller_id, n,
        AVG(L) AS media, MIN(L) AS mn, MAX(L) AS mx,
        -- p25
        MAX(CASE WHEN rn0 = CAST((n-1)*0.25 AS INTEGER)     THEN L END) AS p25_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.25 AS INTEGER) + 1 THEN L END) AS p25_hi,
        (n-1)*0.25 - CAST((n-1)*0.25 AS INTEGER)                        AS p25_f,
        -- p50 (mediana)
        MAX(CASE WHEN rn0 = CAST((n-1)*0.50 AS INTEGER)     THEN L END) AS p50_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.50 AS INTEGER) + 1 THEN L END) AS p50_hi,
        (n-1)*0.50 - CAST((n-1)*0.50 AS INTEGER)                        AS p50_f,
        -- p75
        MAX(CASE WHEN rn0 = CAST((n-1)*0.75 AS INTEGER)     THEN L END) AS p75_lo,
        MAX(CASE WHEN rn0 = CAST((n-1)*0.75 AS INTEGER) + 1 THEN L END) AS p75_hi,
        (n-1)*0.75 - CAST((n-1)*0.75 AS INTEGER)                        AS p75_f
    FROM ranked
    GROUP BY seller_id, n
),
spine AS (SELECT DISTINCT seller_id FROM vendas)
SELECT
    s.seller_id,
    agg.media                                                              AS desc_chars_media,
    CASE WHEN agg.p25_hi IS NULL THEN agg.p25_lo ELSE agg.p25_lo + agg.p25_f*(agg.p25_hi-agg.p25_lo) END AS desc_chars_p25,
    CASE WHEN agg.p50_hi IS NULL THEN agg.p50_lo ELSE agg.p50_lo + agg.p50_f*(agg.p50_hi-agg.p50_lo) END AS desc_chars_mediana,
    CASE WHEN agg.p75_hi IS NULL THEN agg.p75_lo ELSE agg.p75_lo + agg.p75_f*(agg.p75_hi-agg.p75_lo) END AS desc_chars_p75,
    agg.mn                                                                 AS desc_chars_min,
    agg.mx                                                                 AS desc_chars_max
FROM spine s
LEFT JOIN agg ON agg.seller_id = s.seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 6. Peso médio e mediana dos produtos  `*`

Peso (g) **médio** e **mediano** das unidades vendidas, por janela (ponderado por venda).

In [ ]:
# Feature 06_peso_medio_mediana  (fonte: features_sqlite/06_peso_medio_mediana.sql)
SQL = """
-- =====================================================================
-- Feature 06 (variaveis.md item 6) — Peso médio e mediana dos produtos
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : peso_medio_g_{d28,d56,d365,vida}, peso_mediana_g_{...}
-- Definição: média e mediana de product_weight_g das UNIDADES vendidas na
--            janela (ponderado por venda — sem DISTINCT, premissa 3.6).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Mediana  : SQLite não tem MEDIAN; usamos percentil contínuo tipo-7
--            (igual ao percentile(w,0.5) do Spark), ranqueado DENTRO de
--            cada janela. Pesos NULL ignorados; janela sem venda -> NULL.
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        p.product_weight_g          AS w,
        o.order_purchase_timestamp  AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
avgs AS (   -- médias por janela (emite todos os sellers)
    SELECT
        seller_id,
        AVG(CASE WHEN dt_venda >= datetime('{data_corte}', '-28 days')  THEN w END) AS peso_medio_g_d28,
        AVG(CASE WHEN dt_venda >= datetime('{data_corte}', '-56 days')  THEN w END) AS peso_medio_g_d56,
        AVG(CASE WHEN dt_venda >= datetime('{data_corte}', '-365 days') THEN w END) AS peso_medio_g_d365,
        AVG(w)                                                                       AS peso_medio_g_vida
    FROM vendas
    GROUP BY seller_id
),
-- ----- ranqueamento por janela (para a mediana) -----
r_d28  AS (SELECT seller_id, w, CAST(ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY w) AS INTEGER)-1 AS rn0, COUNT(*) OVER (PARTITION BY seller_id) AS n FROM vendas WHERE w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-28 days')),
r_d56  AS (SELECT seller_id, w, CAST(ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY w) AS INTEGER)-1 AS rn0, COUNT(*) OVER (PARTITION BY seller_id) AS n FROM vendas WHERE w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-56 days')),
r_d365 AS (SELECT seller_id, w, CAST(ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY w) AS INTEGER)-1 AS rn0, COUNT(*) OVER (PARTITION BY seller_id) AS n FROM vendas WHERE w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-365 days')),
r_vida AS (SELECT seller_id, w, CAST(ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY w) AS INTEGER)-1 AS rn0, COUNT(*) OVER (PARTITION BY seller_id) AS n FROM vendas WHERE w IS NOT NULL),
-- mediana interpolada (tipo-7) por janela
m_d28  AS (SELECT seller_id, MAX(CASE WHEN rn0=CAST((n-1)*0.5 AS INTEGER) THEN w END) AS lo, MAX(CASE WHEN rn0=CAST((n-1)*0.5 AS INTEGER)+1 THEN w END) AS hi, (n-1)*0.5-CAST((n-1)*0.5 AS INTEGER) AS f FROM r_d28  GROUP BY seller_id, n),
m_d56  AS (SELECT seller_id, MAX(CASE WHEN rn0=CAST((n-1)*0.5 AS INTEGER) THEN w END) AS lo, MAX(CASE WHEN rn0=CAST((n-1)*0.5 AS INTEGER)+1 THEN w END) AS hi, (n-1)*0.5-CAST((n-1)*0.5 AS INTEGER) AS f FROM r_d56  GROUP BY seller_id, n),
m_d365 AS (SELECT seller_id, MAX(CASE WHEN rn0=CAST((n-1)*0.5 AS INTEGER) THEN w END) AS lo, MAX(CASE WHEN rn0=CAST((n-1)*0.5 AS INTEGER)+1 THEN w END) AS hi, (n-1)*0.5-CAST((n-1)*0.5 AS INTEGER) AS f FROM r_d365 GROUP BY seller_id, n),
m_vida AS (SELECT seller_id, MAX(CASE WHEN rn0=CAST((n-1)*0.5 AS INTEGER) THEN w END) AS lo, MAX(CASE WHEN rn0=CAST((n-1)*0.5 AS INTEGER)+1 THEN w END) AS hi, (n-1)*0.5-CAST((n-1)*0.5 AS INTEGER) AS f FROM r_vida GROUP BY seller_id, n)
SELECT
    a.seller_id,
    a.peso_medio_g_d28,
    CASE WHEN m_d28.hi  IS NULL THEN m_d28.lo  ELSE m_d28.lo  + m_d28.f *(m_d28.hi -m_d28.lo)  END AS peso_mediana_g_d28,
    a.peso_medio_g_d56,
    CASE WHEN m_d56.hi  IS NULL THEN m_d56.lo  ELSE m_d56.lo  + m_d56.f *(m_d56.hi -m_d56.lo)  END AS peso_mediana_g_d56,
    a.peso_medio_g_d365,
    CASE WHEN m_d365.hi IS NULL THEN m_d365.lo ELSE m_d365.lo + m_d365.f*(m_d365.hi-m_d365.lo) END AS peso_mediana_g_d365,
    a.peso_medio_g_vida,
    CASE WHEN m_vida.hi IS NULL THEN m_vida.lo ELSE m_vida.lo + m_vida.f*(m_vida.hi-m_vida.lo) END AS peso_mediana_g_vida
FROM avgs a
LEFT JOIN m_d28  ON m_d28.seller_id  = a.seller_id
LEFT JOIN m_d56  ON m_d56.seller_id  = a.seller_id
LEFT JOIN m_d365 ON m_d365.seller_id = a.seller_id
LEFT JOIN m_vida ON m_vida.seller_id = a.seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 7. Peso total dos produtos  `*`

Massa **total (kg)** despachada por janela. Variável de **escala** da operação.

In [ ]:
# Feature 07_peso_total  (fonte: features_sqlite/07_peso_total.sql)
SQL = """
-- =====================================================================
-- Feature 07 (variaveis.md item 7) — Peso total dos produtos (kg)
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : peso_total_kg_{d28,d56,d365,vida}
-- Definição: massa total (kg) de TODAS as unidades vendidas na janela.
-- Data     : order_purchase_timestamp < {data_corte}.
-- Premissas: 1 linha de order_items = 1 unidade -> SUM já é ponderado;
--            convertido g->kg (/1000); pesos NULL não contribuem; janela
--            sem venda -> NULL.
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        p.product_weight_g          AS w,
        o.order_purchase_timestamp  AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT
    seller_id,
    SUM(CASE WHEN dt_venda >= datetime('{data_corte}', '-28 days')  THEN w END) / 1000.0 AS peso_total_kg_d28,
    SUM(CASE WHEN dt_venda >= datetime('{data_corte}', '-56 days')  THEN w END) / 1000.0 AS peso_total_kg_d56,
    SUM(CASE WHEN dt_venda >= datetime('{data_corte}', '-365 days') THEN w END) / 1000.0 AS peso_total_kg_d365,
    SUM(w) / 1000.0                                                                       AS peso_total_kg_vida
FROM vendas
GROUP BY seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 8. Cubagem média dos produtos  `*`

Volume **médio** (cm³ = `L×H×W`) das unidades vendidas por janela.

In [ ]:
# Feature 08_cubagem_media  (fonte: features_sqlite/08_cubagem_media.sql)
SQL = """
-- =====================================================================
-- Feature 08 (variaveis.md item 8) — Cubagem média dos produtos (cm³)
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : cubagem_media_cm3_{d28,d56,d365,vida}
-- Definição: volume médio (L×H×W da caixa de envio) das UNIDADES vendidas
--            na janela (ponderado por venda).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Premissas: cubagem = length_cm*height_cm*width_cm; qualquer dimensão
--            NULL -> cubagem NULL -> ignorada no AVG; janela sem venda -> NULL.
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        (p.product_length_cm * p.product_height_cm * p.product_width_cm) AS cub,
        o.order_purchase_timestamp                                       AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT
    seller_id,
    AVG(CASE WHEN dt_venda >= datetime('{data_corte}', '-28 days')  THEN cub END) AS cubagem_media_cm3_d28,
    AVG(CASE WHEN dt_venda >= datetime('{data_corte}', '-56 days')  THEN cub END) AS cubagem_media_cm3_d56,
    AVG(CASE WHEN dt_venda >= datetime('{data_corte}', '-365 days') THEN cub END) AS cubagem_media_cm3_d365,
    AVG(cub)                                                                       AS cubagem_media_cm3_vida
FROM vendas
GROUP BY seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 9. Cubagem total dos produtos  `*`

Volume **total** (cm³) despachado por janela. Pareado com o peso total, descreve a escala logística.

In [ ]:
# Feature 09_cubagem_total  (fonte: features_sqlite/09_cubagem_total.sql)
SQL = """
-- =====================================================================
-- Feature 09 (variaveis.md item 9) — Cubagem total dos produtos (cm³)
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : cubagem_total_cm3_{d28,d56,d365,vida}
-- Definição: volume total enviado (soma das cubagens de cada unidade) na
--            janela. Pareado com peso_total descreve a "escala" do seller.
-- Data     : order_purchase_timestamp < {data_corte}.
-- Premissas: cubagem = length_cm*height_cm*width_cm; dimensão NULL ->
--            cubagem NULL -> não contribui; janela sem venda -> NULL.
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        (p.product_length_cm * p.product_height_cm * p.product_width_cm) AS cub,
        o.order_purchase_timestamp                                       AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT
    seller_id,
    SUM(CASE WHEN dt_venda >= datetime('{data_corte}', '-28 days')  THEN cub END) AS cubagem_total_cm3_d28,
    SUM(CASE WHEN dt_venda >= datetime('{data_corte}', '-56 days')  THEN cub END) AS cubagem_total_cm3_d56,
    SUM(CASE WHEN dt_venda >= datetime('{data_corte}', '-365 days') THEN cub END) AS cubagem_total_cm3_d365,
    SUM(cub)                                                                       AS cubagem_total_cm3_vida
FROM vendas
GROUP BY seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 10. Quantidade média de fotos por produto  (Vida)

Média de `product_photos_qty` entre os **produtos distintos** vendidos. Proxy de qualidade da vitrine.

In [ ]:
# Feature 10_fotos_media_por_produto  (fonte: features_sqlite/10_fotos_media_por_produto.sql)
SQL = """
-- =====================================================================
-- Feature 10 (variaveis.md item 10) — Qtd média de fotos por produto
-- Janela: Vida (sem '*')           |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Coluna   : qtd_fotos_media_por_produto
-- Definição: média de product_photos_qty entre os PRODUTOS DISTINTOS
--            vendidos pelo seller (cada produto pesa 1 — propriedade de
--            cadastro, premissa 3.6).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Nulos    : fotos NULL ignoradas; seller sem produto com fotos -> NULL
--            (mantido via spine).
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.product_id,
        p.product_photos_qty AS fotos
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
),
prod AS (   -- produtos DISTINTOS do seller
    SELECT DISTINCT seller_id, product_id, fotos
    FROM vendas
),
agg AS (
    SELECT seller_id, AVG(fotos) AS qtd_fotos_media_por_produto
    FROM prod
    GROUP BY seller_id
),
spine AS (SELECT DISTINCT seller_id FROM vendas)
SELECT
    s.seller_id,
    agg.qtd_fotos_media_por_produto
FROM spine s
LEFT JOIN agg ON agg.seller_id = s.seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 11. Preço por kg  `*`

`SUM(price) / SUM(kg)` por janela (R$/kg). **Valor agregado** praticado.

In [ ]:
# Feature 11_preco_por_kg  (fonte: features_sqlite/11_preco_por_kg.sql)
SQL = """
-- =====================================================================
-- Feature 11 (variaveis.md item 11) — Preço por kg (R$/kg)
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : preco_por_kg_{d28,d56,d365,vida}
-- Definição: receita total / massa total no período = SUM(price) / SUM(kg).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Premissas: receita = price (NÃO inclui frete — ver item 12); numerador e
--            denominador restritos aos itens com peso não-nulo (mesma base);
--            denominador 0/NULL -> NULL (NULLIF). price é por unidade (grão
--            da fato), então SUM(price) = receita total.
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.price                    AS price,
        p.product_weight_g          AS w,
        o.order_purchase_timestamp  AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT
    seller_id,
    SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-28 days')  THEN price END)
      / NULLIF(SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-28 days')  THEN w END) / 1000.0, 0) AS preco_por_kg_d28,
    SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-56 days')  THEN price END)
      / NULLIF(SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-56 days')  THEN w END) / 1000.0, 0) AS preco_por_kg_d56,
    SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-365 days') THEN price END)
      / NULLIF(SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-365 days') THEN w END) / 1000.0, 0) AS preco_por_kg_d365,
    SUM(CASE WHEN w IS NOT NULL THEN price END)
      / NULLIF(SUM(CASE WHEN w IS NOT NULL THEN w END) / 1000.0, 0)                                                       AS preco_por_kg_vida
FROM vendas
GROUP BY seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 12. Custo de frete por kg  `*`

`SUM(freight_value) / SUM(kg)` por janela (R$/kg). **Custo logístico**.

In [ ]:
# Feature 12_frete_por_kg  (fonte: features_sqlite/12_frete_por_kg.sql)
SQL = """
-- =====================================================================
-- Feature 12 (variaveis.md item 12) — Custo de frete por kg (R$/kg)
-- Janelas: D28, D56, D365, Vida   |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : frete_por_kg_{d28,d56,d365,vida}
-- Definição: frete total / massa total = SUM(freight_value) / SUM(kg).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Premissas: freight_value já vem alocado por item pela Olist (sem rateio
--            próprio); numerador/denominador restritos a peso não-nulo;
--            denominador 0/NULL -> NULL (NULLIF).
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH vendas AS (
    SELECT
        oi.seller_id,
        oi.freight_value            AS frete,
        p.product_weight_g          AS w,
        o.order_purchase_timestamp  AS dt_venda
    FROM order_items oi
    JOIN orders   o ON o.order_id   = oi.order_id
    JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
)
SELECT
    seller_id,
    SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-28 days')  THEN frete END)
      / NULLIF(SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-28 days')  THEN w END) / 1000.0, 0) AS frete_por_kg_d28,
    SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-56 days')  THEN frete END)
      / NULLIF(SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-56 days')  THEN w END) / 1000.0, 0) AS frete_por_kg_d56,
    SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-365 days') THEN frete END)
      / NULLIF(SUM(CASE WHEN w IS NOT NULL AND dt_venda >= datetime('{data_corte}', '-365 days') THEN w END) / 1000.0, 0) AS frete_por_kg_d365,
    SUM(CASE WHEN w IS NOT NULL THEN frete END)
      / NULLIF(SUM(CASE WHEN w IS NOT NULL THEN w END) / 1000.0, 0)                                                       AS frete_por_kg_vida
FROM vendas
GROUP BY seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 13. Top 3 categorias do vendedor  (Vida)

As **3 categorias de maior receita** (`SUM(price)`) do seller. Posições inexistentes ficam `NULL`.

In [ ]:
# Feature 13_top3_categorias  (fonte: features_sqlite/13_top3_categorias.sql)
SQL = """
-- =====================================================================
-- Feature 13 (variaveis.md item 13) — Top 3 categorias do vendedor
-- Janela: Vida (sem `*`)           |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : top1_categoria, top2_categoria, top3_categoria (nomes em pt)
-- Definição: as 3 categorias com maior receita acumulada (SUM(price)) até
--            {data_corte}.
-- Data     : order_purchase_timestamp < {data_corte}.
-- Desempate: (1) maior SUM(price); (2) maior nº de pedidos distintos;
--            (3) ordem alfabética da categoria (premissa 3.8).
-- Nulos    : categoria NULL -> 'sem_categoria'; seller com <3 categorias
--            -> posições inexistentes ficam NULL.
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH cat_rev AS (
    SELECT
        oi.seller_id,
        COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
        SUM(oi.price)                AS receita,
        COUNT(DISTINCT oi.order_id)  AS pedidos
    FROM order_items oi
    JOIN orders      o ON o.order_id   = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
    GROUP BY oi.seller_id, categoria
),
ranked AS (
    SELECT
        seller_id, categoria,
        ROW_NUMBER() OVER (
            PARTITION BY seller_id
            ORDER BY receita DESC, pedidos DESC, categoria ASC
        ) AS rk
    FROM cat_rev
)
SELECT
    seller_id,
    MAX(CASE WHEN rk = 1 THEN categoria END) AS top1_categoria,
    MAX(CASE WHEN rk = 2 THEN categoria END) AS top2_categoria,
    MAX(CASE WHEN rk = 3 THEN categoria END) AS top3_categoria
FROM ranked
GROUP BY seller_id;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

## 14. Share das top 3 categorias  (Vida)

**Fração da receita** concentrada nas 3 maiores categorias. Mede **concentração** (fragilidade).

In [ ]:
# Feature 14_share_top3_categorias  (fonte: features_sqlite/14_share_top3_categorias.sql)
SQL = """
-- =====================================================================
-- Feature 14 (variaveis.md item 14) — Share das top 3 categorias
-- Janela: Vida (sem `*`)           |   Grão de saída: 1 linha por seller_id
-- Dialeto: SQLite (validação local)
-- ---------------------------------------------------------------------
-- Colunas  : share_top1, share_top2, share_top3  (frações em [0,1])
-- Definição: fração da receita total (Vida) concentrada em cada uma das 3
--            categorias de maior receita (mesmo ranking do item 13).
-- Data     : order_purchase_timestamp < {data_corte}.
-- Premissas: share_topk = receita(topk) / receita_total; mesma regra de
--            desempate do item 13; soma dos 3 shares <= 1.
-- Nulos    : seller com <k categorias -> share_topk NULL; receita_total 0
--            -> NULL (NULLIF).
-- Parâmetro: {data_corte} (ex. 2018-09-01).
-- =====================================================================
WITH cat_rev AS (
    SELECT
        oi.seller_id,
        COALESCE(p.product_category_name, 'sem_categoria') AS categoria,
        SUM(oi.price)                AS receita,
        COUNT(DISTINCT oi.order_id)  AS pedidos
    FROM order_items oi
    JOIN orders      o ON o.order_id   = oi.order_id
    LEFT JOIN products p ON p.product_id = oi.product_id
    WHERE o.order_purchase_timestamp < '{data_corte}'
    GROUP BY oi.seller_id, categoria
),
ranked AS (
    SELECT
        seller_id, categoria, receita,
        ROW_NUMBER() OVER (
            PARTITION BY seller_id
            ORDER BY receita DESC, pedidos DESC, categoria ASC
        ) AS rk
    FROM cat_rev
),
tot AS (
    SELECT seller_id, SUM(receita) AS receita_total
    FROM cat_rev
    GROUP BY seller_id
)
SELECT
    r.seller_id,
    MAX(CASE WHEN r.rk = 1 THEN r.receita END) / NULLIF(t.receita_total, 0) AS share_top1,
    MAX(CASE WHEN r.rk = 2 THEN r.receita END) / NULLIF(t.receita_total, 0) AS share_top2,
    MAX(CASE WHEN r.rk = 3 THEN r.receita END) / NULLIF(t.receita_total, 0) AS share_top3
FROM ranked r
JOIN tot t ON t.seller_id = r.seller_id
GROUP BY r.seller_id, t.receita_total;
"""

df = run_sql(SQL)
print(f'{len(df):,} sellers x {len(df.columns)} colunas')
df.head(10)

---

### Pronto

Cada tabela acima é o resultado da variável para todos os sellers ativos no corte. Para a tabela larga única, basta um `LEFT JOIN` dos 14 resultados por `seller_id`.